# 04 — Reliability Analysis

## Objective

Evaluate equipment reliability before and after LSS using:

- Failure frequency
- MTTR (Mean Time to Repair)
- MTBF (Mean Time Between Failures)
- Failure-mode comparison

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [3]:
downtime_before = pd.read_csv(RAW_DIR / "DowntimeDataset.csv")
downtime_after = pd.read_csv(RAW_DIR / "DowntimeDataset_afterLSS.csv")

oee_before = pd.read_csv(RAW_DIR / "OEEdataset.csv")
oee_after = pd.read_csv(RAW_DIR / "OEEdataset_afterLSS.csv")

for df in [downtime_before, downtime_after, oee_before, oee_after]:
    df.dropna(axis=1, how="all", inplace=True)
    df.dropna(axis=0, how="all", inplace=True)

C:\Users\erenk\AppData\Local\Temp\ipykernel_21788\3423606821.py:2: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  downtime_after = pd.read_csv(RAW_DIR / "DowntimeDataset_afterLSS.csv")


In [4]:
downtime_before = downtime_before.rename(columns={
    "StopGroup": "stop_group",
    "Stop": "stop",
    "StopDuration(min)": "duration_min"
})

downtime_after = downtime_after.rename(columns={
    "Stop group": "stop_group",
    "Stop": "stop",
    "Stop duration (min) (TTR)": "duration_min"
})

oee_before = oee_before.rename(columns={
    "Sumof goodProductionTimeSec": "good_production_time_sec",
    "Sum of slowProductionTimeSec": "slow_production_time_sec"
})

oee_after = oee_after.rename(columns={
    "Sum of goodProdction": "good_production_time_sec",
    "Sum of slowProduction": "slow_production_time_sec"
})

In [5]:
failure_before = downtime_before[
    downtime_before["stop_group"] == "FAILURE"
].copy()

failure_after = downtime_after[
    downtime_after["stop_group"] == "FAILURE"
].copy()

In [6]:
runtime_before_hours = (
    oee_before["good_production_time_sec"].sum()
    + oee_before["slow_production_time_sec"].sum()
) / 3600

runtime_after_hours = (
    oee_after["good_production_time_sec"].sum()
    + oee_after["slow_production_time_sec"].sum()
) / 3600


reliability_summary = pd.DataFrame({
    "period": ["Before LSS", "After LSS"],
    "failure_events": [
        len(failure_before),
        len(failure_after)
    ],
    "runtime_hours": [
        runtime_before_hours,
        runtime_after_hours
    ],
    "failure_downtime_min": [
        failure_before["duration_min"].sum(),
        failure_after["duration_min"].sum()
    ],
    "MTTR_min": [
        failure_before["duration_min"].mean(),
        failure_after["duration_min"].mean()
    ],
    "MTBF_hours": [
        runtime_before_hours / len(failure_before),
        runtime_after_hours / len(failure_after)
    ]
})

reliability_summary

,period,failure_events,runtime_hours,failure_downtime_min,MTTR_min,MTBF_hours
0,Before LSS,222,326.006667,3233.6,14.565766,1.468498
1,After LSS,116,223.270000,1690.0,14.568966,1.924741


### Reliability Findings

Failure frequency decreased substantially after LSS.

Approximate MTBF increased from 1.47 to 1.92 operating hours, indicating improved equipment reliability. In contrast, MTTR remained almost unchanged at approximately 14.6 minutes.

The improvement therefore appears to come primarily from fewer failures rather than faster repair.

In [7]:
failure_mode_summary = (
    pd.concat([
        failure_before.assign(period="Before LSS"),
        failure_after.assign(period="After LSS")
    ])
    .groupby(["period", "stop"])
    .agg(
        failure_events=("duration_min", "size"),
        downtime_min=("duration_min", "sum"),
        mttr_min=("duration_min", "mean")
    )
    .reset_index()
)

failure_mode_summary

,period,stop,failure_events,downtime_min,mttr_min
0,After LSS,AUTOMATION FAILURE,23,429.0,18.652174
1,After LSS,MECHANICAL FAILURE,35,818.0,23.371429
2,After LSS,OTHER FAILURE,35,334.0,9.542857
3,After LSS,WIRE FAILURE,23,109.0,4.739130
4,Before LSS,AUTOMATION FAILURE,131,1763.3,13.460305
5,Before LSS,MECHANICAL FAILURE,89,1451.0,16.303371
6,Before LSS,OTHER FAILURE,2,19.3,9.650000


In [8]:
runtime_map = {
    "Before LSS": runtime_before_hours,
    "After LSS": runtime_after_hours
}

failure_mode_summary["runtime_hours"] = (
    failure_mode_summary["period"].map(runtime_map)
)

failure_mode_summary["failures_per_100h"] = (
    failure_mode_summary["failure_events"]
    / failure_mode_summary["runtime_hours"]
    * 100
)

failure_mode_summary.sort_values(
    ["period", "failures_per_100h"],
    ascending=[True, False]
)

,period,stop,failure_events,downtime_min,mttr_min,runtime_hours,failures_per_100h
1,After LSS,MECHANICAL FAILURE,35,818.0,23.371429,223.270000,15.676087
2,After LSS,OTHER FAILURE,35,334.0,9.542857,223.270000,15.676087
0,After LSS,AUTOMATION FAILURE,23,429.0,18.652174,223.270000,10.301429
3,After LSS,WIRE FAILURE,23,109.0,4.739130,223.270000,10.301429
4,Before LSS,AUTOMATION FAILURE,131,1763.3,13.460305,326.006667,40.183227
5,Before LSS,MECHANICAL FAILURE,89,1451.0,16.303371,326.006667,27.300055
6,Before LSS,OTHER FAILURE,2,19.3,9.650000,326.006667,0.613484


### Failure Mode Findings

Overall equipment reliability improved after LSS, as reflected by the higher MTBF.

Automation and Mechanical Failure frequencies declined substantially relative to operating time. However, new or increasingly important failure modes emerged, particularly Other Failure and Wire Failure.

Repair duration did not improve overall, and MTTR increased for some major failure categories. This indicates that future improvement efforts should address both emerging failure modes and repair effectiveness.